In [3]:
import sys

from pathlib import Path

parent_folder = Path.cwd().parent
if str(parent_folder) not in sys.path:
    sys.path.append(str(parent_folder))
    
import warnings
warnings.filterwarnings('ignore')
import numpy as np

import simulation_functional as sim
import xobjects as xo

In [31]:
def aa_to_canon(line, sqrt_action, phase, axis='x'):
    tw = line.twiss(method='4d', freeze_longitudinal=True)
    if axis=='x':
        beta = tw.betx[0]
        alpha = tw.alfx[0]
    elif axis=='y':
        beta = tw.bety[0]
        alpha = tw.alfy[0]
        
    q_norm = np.sqrt(2)*sqrt_action * np.cos(phase)
    p_norm = -np.sqrt(2)*sqrt_action * np.sin(phase)
    q = q_norm * np.sqrt(beta)
    p = (p_norm - alpha * q_norm) / np.sqrt(beta)
    return q, p

In [48]:
gemitt=60e-6
def run_all(pipesize, n_turns=1000):
    line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))
    
    phase = 0
    n=50
    sqrt_Jx_max = np.sqrt(4.5*gemitt)
    sqrt_Jx_list = np.linspace(sqrt_Jx_max/n, sqrt_Jx_max, n)
    x, px = aa_to_canon(line, sqrt_Jx_list, phase)
    print(x)
    sqrt_Jy_max = np.sqrt(4.5*gemitt)
    sqrt_Jy_list = np.linspace(sqrt_Jy_max/n, sqrt_Jy_max, n)
    y, py = aa_to_canon(line, sqrt_Jy_list, phase, axis='y')
    XX, YY = np.meshgrid(x, y)
    PX, PY = np.meshgrid(px, py)
    
    tw = line.twiss(method='4d', freeze_longitudinal=True)
    particles = line.build_particles(
        x=XX.flatten()+tw.x[0],
        y=YY.flatten()+tw.y[0],
        px=PX.flatten()+tw.px[0],
        py=PY.flatten()+tw.py[0],
        method='4d', freeze_longitudinal=True)
    
    line = sim.add_monitors(line, particles, n_monitors=1, n_turns=n_turns)
    twiss = line.twiss(method='4d', freeze_longitudinal=True)

    if pipesize=='none':
        line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=5e12,
                                                     n_interactions=100, 
                                                     gemitt=gemitt, indirect=True,
                                                     pipe_enlargement=20)
    else:
        line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=5e12,
                                                     n_interactions=100, 
                                                     gemitt=gemitt, indirect=True,
                                                     pipe_enlargement=pipesize)
    
    line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
    line.track(particles, num_turns=n_turns, time=True)
    sim.save(line, particles, twiss, 
             filename=f'data/3sig_pipesize:{pipesize}', 
             aper_df=aper_df, beam_df=beam_df)

In [49]:
#for pipesize in ['none', 0.5, 0.75, 1, 1.5, 2, 3]:
for pipesize in ['none']:#0.6, 0.7, 0.8, 0.9, 1, 1.2, 1.5, 2]:
    run_all(pipesize=pipesize)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
[0.00151209 0.00302418 0.00453627 0.00604837 0.00756046 0.00907255
 0.01058464 0.01209673 0.01360882 0.01512092 0.01663301 0.0181451
 0.01965719 0.02116928 0.02268137 0.02419347 0.02570556 0.02721765
 0.02872974 0.03024183 0.03175392 0.03326601 0.03477811 0.0362902
 0.03780229 0.03931438 0.04082647 0.04233856 0.04385066 0.04536275
 0.04687484 0.04838693 0.04989902 0.05141111 0.0529232  0.0544353
 0.05594739 0.05745948 0.05897157 0.06048366 0.06199575 0.06350785
 0.06501994 0.06653203 0.06804412 0.06955621 0.0710683  0.0725804
 0.07409249 0.07560458]


Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3981 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/3sig_pipesize:none


# Matched beam

In [52]:
gemitt=60e-6
n_turns=1000
n_particles=10000
for indirect in [False, True]:
    line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))

    ref = line.particle_ref
    bg = ref.beta0 * ref.gamma0
    tw = line.twiss(method='4d', freeze_longitudinal=True)
    particles = line.build_particles(
        x_norm=np.random.normal(size=n_particles),
        y_norm=np.random.normal(size=n_particles),
        px_norm=np.random.normal(size=n_particles),
        py_norm=np.random.normal(size=n_particles),
        nemitt_x=bg*gemitt,
        nemitt_y=bg*gemitt,
        method='4d',
        freeze_longitudinal=True,
        mode='normalized_transverse')
    particles.x += (tw.x[0] - np.mean(particles.x))
    particles.y += (tw.y[0] - np.mean(particles.y))
    particles.px += (tw.px[0] - np.mean(particles.px))
    particles.py += (tw.py[0] - np.mean(particles.py))

    
    line = sim.add_monitors(line, particles, n_monitors=1, n_turns=n_turns)
    twiss = line.twiss(method='4d', freeze_longitudinal=True)


    line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=5e12,
                                                 n_interactions=100, 
                                                 gemitt=gemitt, indirect=indirect)
    
    line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
    line.track(particles, num_turns=n_turns, time=True)
    sim.save(line, particles, twiss, 
             filename=f'data/matched_indirect:{indirect}', 
             aper_df=aper_df, beam_df=beam_df)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3981 [00:00<?, ?it/s]

...

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3981 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/matched_indirect:True
